In [4]:
from pathlib import Path

folders = ["data/raw", "data/processed", "notebooks", "src", "report"]

for f in folders:
    p = Path(f)
    p.mkdir(parents=True, exist_ok=True)
    (p / ".gitkeep").touch()

print("Created:", folders)

Created: ['data/raw', 'data/processed', 'notebooks', 'src', 'report']


In [5]:
!git add .
!git commit -m "Add project folder structure"
!git push

[main 65c4c7c] Add project folder structure
 5 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 data/processed/.gitkeep
 create mode 100644 data/raw/.gitkeep
 create mode 100644 notebooks/.gitkeep
 create mode 100644 report/.gitkeep
 create mode 100644 src/.gitkeep


To https://github.com/l253042/E-commerce.git
   44fbba1..65c4c7c  main -> main


In [9]:
from pathlib import Path
Path("src/config.py").write_text("SEED = 3020\n")


12

In [10]:
import sys
from pathlib import Path

# find the repo's src folder whether you're in the root or in notebooks/
src = Path("src") if Path("src").exists() else Path("../src")
sys.path.append(str(src.resolve()))

from config import SEED
print(SEED)

3020


In [11]:
!git add src/config.py
!git commit -m "Add config with SEED"
!git push

[main 77065b7] Add config with SEED
 1 file changed, 1 insertion(+)
 create mode 100644 src/config.py


To https://github.com/l253042/E-commerce.git
   65c4c7c..77065b7  main -> main


In [12]:
from pathlib import Path

Path("requirements.txt").write_text(
"""requests
beautifulsoup4
pandas
numpy
scikit-learn
matplotlib
seaborn
scipy
"""
)

75

In [14]:
from sklearn.feature_selection import SequentialFeatureSelector
!pip install -r requirements.txt


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
!git add requirements.txt
!git commit -m "Add requirements.txt"
!git push

[main 89b91a5] Add requirements.txt
 1 file changed, 8 insertions(+)
 create mode 100644 requirements.txt


To https://github.com/l253042/E-commerce.git
   77065b7..89b91a5  main -> main


In [16]:
import sys, time, hashlib, requests
import pandas as pd
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin
from bs4 import BeautifulSoup

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = ROOT / "data" / "raw" / "books_raw.csv"

sys.path.append(str(ROOT / "src"))
from config import SEED
print("SEED =", SEED)

SEED = 3020


In [18]:
START_URL = "https://books.toscrape.com/catalogue/page-1.html"
SLEEP = 0.5
MAX_RETRIES = 3

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (university project scraper)"})
request_count = 0
failed_urls = []

def get_soup(url):
    global request_count
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            request_count += 1
            r = session.get(url, timeout=15)
            r.raise_for_status()
            r.encoding = "utf-8"
            time.sleep(SLEEP)
            return BeautifulSoup(r.text, "html.parser")
        except requests.RequestException as e:
            print(f"[retry {attempt}/{MAX_RETRIES}] {url} -> {e}")
            time.sleep(2 * attempt)
    failed_urls.append(url)
    return None

def get_product_urls():
    urls, pages, page_url = [], 0, START_URL
    while page_url:
        soup = get_soup(page_url)
        if soup is None:
            break
        pages += 1
        urls += [urljoin(page_url, a["href"]) for a in soup.select("article.product_pod h3 a")]
        nxt = soup.select_one("li.next a")
        page_url = urljoin(page_url, nxt["href"]) if nxt else None
    return urls, pages

def parse_product(url):
    soup = get_soup(url)
    if soup is None:
        return None
    table = {tr.th.get_text(strip=True): tr.td.get_text(strip=True)
             for tr in soup.select("table.table-striped tr")}
    crumbs = soup.select("ul.breadcrumb li")
    desc_header = soup.select_one("#product_description")
    desc = desc_header.find_next_sibling("p") if desc_header else None
    rating = soup.select_one("p.star-rating")
    return {
        "title": soup.h1.get_text(strip=True),
        "product_url": url,
        "category": crumbs[2].get_text(strip=True) if len(crumbs) > 2 else None,
        "price_excl_tax": table.get("Price (excl. tax)"),
        "price_incl_tax": table.get("Price (incl. tax)"),
        "tax": table.get("Tax"),
        "rating": rating["class"][1] if rating else None,
        "availability": table.get("Availability"),
        "num_reviews": table.get("Number of reviews"),
        "upc": table.get("UPC"),
        "description": desc.get_text(strip=True) if desc else None,
        "collected_at": datetime.now().isoformat(timespec="seconds"),
    }

In [19]:
soup = get_soup(START_URL)
print("Books on page 1:", len(soup.select("article.product_pod")))
print("Next-page link:", soup.select_one("li.next a")["href"])

cats = soup.select("div.side_categories ul li ul li a")
print("Categories in sidebar:", len(cats))

first = urljoin(START_URL, soup.select_one("article.product_pod h3 a")["href"])
print("Example product page:", first)
print(parse_product(first))

Books on page 1: 20
Next-page link: page-2.html
Categories in sidebar: 50
Example product page: https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
{'title': 'A Light in the Attic', 'product_url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html', 'category': 'Poetry', 'price_excl_tax': '£51.77', 'price_incl_tax': '£51.77', 'tax': '£0.00', 'rating': 'Three', 'availability': 'In stock (22 available)', 'num_reviews': '0', 'upc': 'a897fe39b1053632', 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings

In [20]:
collection_start = datetime.now()

product_urls, n_pages = get_product_urls()
product_urls = list(dict.fromkeys(product_urls))
print("Listing pages visited:", n_pages, "| unique product URLs:", len(product_urls))

rows = []
for i, url in enumerate(product_urls, 1):
    row = parse_product(url)
    if row:
        rows.append(row)
    if i % 100 == 0:
        print(f"{i}/{len(product_urls)} done | requests so far: {request_count}")

collection_end = datetime.now()

df = pd.DataFrame(rows)
RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(RAW_PATH, index=False, encoding="utf-8", lineterminator="\n")
print("Saved", len(df), "rows to", RAW_PATH)
print("Failed URLs:", failed_urls)

Listing pages visited: 50 | unique product URLs: 1000
100/1000 done | requests so far: 152
200/1000 done | requests so far: 252
300/1000 done | requests so far: 352
400/1000 done | requests so far: 452
500/1000 done | requests so far: 552
600/1000 done | requests so far: 652
700/1000 done | requests so far: 752
800/1000 done | requests so far: 852
900/1000 done | requests so far: 952
1000/1000 done | requests so far: 1052
Saved 1000 rows to C:\Users\DELL\data\E-commerce\data\raw\books_raw.csv
Failed URLs: []


In [21]:
ROLL_NUMBERS = ["25L-3042", "25L-3020"]

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

print("Team roll numbers     :", ROLL_NUMBERS)
print("Source URL            :", "https://books.toscrape.com/")
print("Collection start      :", collection_start.isoformat(timespec="seconds"))
print("Collection end        :", collection_end.isoformat(timespec="seconds"))
print("Requests made         :", request_count)
print("Listing pages visited :", n_pages)
print("Rows collected        :", len(df))
print("collected_at filled   :", df["collected_at"].notna().all())
print("SHA-256 of raw file   :", sha256_of(RAW_PATH))

Team roll numbers     : ['25L-3042', '25L-3020']
Source URL            : https://books.toscrape.com/
Collection start      : 2026-11-19T07:35:25
Collection end        : 2026-11-19T07:50:41
Requests made         : 1052
Listing pages visited : 50
Rows collected        : 1000
collected_at filled   : True
SHA-256 of raw file   : f6129f4d7f32af7fc7dff8eef0fdf57c6aca2051987da258d5dab4fa2b0db802


In [22]:
raw = pd.read_csv(RAW_PATH)
print("Rows:", len(raw))
print("Unique product URLs:", raw["product_url"].nunique(), "| Unique UPCs:", raw["upc"].nunique())
print("\nMissing values per column:\n", raw.isna().sum())
print("\nRatings:", raw["rating"].value_counts().to_dict())
print("\nAvailability formats:",
      raw["availability"].str.replace(r"\d+", "N", regex=True).value_counts().to_dict())

Rows: 1000
Unique product URLs: 1000 | Unique UPCs: 1000

Missing values per column:
 title             0
product_url       0
category          0
price_excl_tax    0
price_incl_tax    0
tax               0
rating            0
availability      0
num_reviews       0
upc               0
description       2
collected_at      0
dtype: int64

Ratings: {'One': 226, 'Three': 203, 'Five': 196, 'Two': 196, 'Four': 179}

Availability formats: {'In stock (N available)': 1000}


In [23]:
cat_counts = {}
for c in cats:
    name, url = c.get_text(strip=True), urljoin(START_URL, c["href"])
    n = 0
    while url:
        s = get_soup(url)
        if s is None:
            break
        n += len(s.select("article.product_pod"))
        nxt = s.select_one("li.next a")
        url = urljoin(url, nxt["href"]) if nxt else None
    cat_counts[name] = n

check = pd.DataFrame({"category_pages": cat_counts,
                      "scraped": raw["category"].value_counts().to_dict()}).fillna(0).astype(int)
print("All categories match:", (check["category_pages"] == check["scraped"]).all())

All categories match: True


In [25]:
%cd C:\Users\DELL\data\E-commerce

from pathlib import Path
Path(".gitignore").write_text("__pycache__/\n*.pyc\n.ipynb_checkpoints/\n")

!git rm -r --cached src/__pycache__
!git add .gitignore
!git commit -m "Remove pycache and add .gitignore"
!git push

C:\Users\DELL\data\E-commerce
rm 'src/__pycache__/config.cpython-314.pyc'
[main 2bc2359] Remove pycache and add .gitignore
 2 files changed, 3 insertions(+)
 create mode 100644 .gitignore
 delete mode 100644 src/__pycache__/config.cpython-314.pyc


To https://github.com/l253042/E-commerce.git
   462b75b..2bc2359  main -> main
